# Modelo predictivo Mundial 2026 (versión calibrada)

Este notebook usa el paquete `modelo/` (el mismo que `python simulator.py todo`):

1. **Elo por importancia de torneo** sobre toda la historia (1872 → hoy).
2. **Goles Dixon-Coles** con parámetros estimados por máxima verosimilitud.
3. **Corrección por confederación** y **nitidez en eliminatorias**, aprendidas de los Mundiales 1986-2026 y validadas fuera de muestra.
4. **Monte Carlo** del formato real (48 equipos, mejores terceros, Partidos 73-104), con opción de fijar resultados reales.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from simulator import Context
from modelo.tournament import TournamentSimulator
from modelo.evaluate import match_backtest, summarize_backtest, tournament_evolution

ctx = Context()
pre, post = ctx.pre_model, ctx.post_model
print('Modelo pre-Mundial entrenado hasta', pre.cutoff, '| modelo final hasta', post.cutoff)

## 1. Probabilidades pre-Mundial

In [ ]:
probs = TournamentSimulator(pre, group_fixtures=ctx.fixtures, seed=42).run(10000)
fig = px.bar(probs.head(15), x='Campeón %', y='team', orientation='h', text='Campeón %',
             title='Probabilidad de campeón antes del Mundial (%)')
fig.update_traces(texttemplate='%{text:.1f}%')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)
fig.show()
probs.head(20).round(1)

## 2. Backtest: modelo original vs. nuevo sobre los 104 partidos reales

In [ ]:
bt = match_backtest(ctx.results, ctx.shootouts, ctx.wc, pre)
summary = summarize_backtest(bt)
summary.round(3)

## 3. Evolución de la probabilidad de campeón durante el torneo

In [ ]:
evo = tournament_evolution(pre, ctx.wc, ctx.fixtures, n_sims=3000)
top = evo.head(6)
fig = go.Figure()
for team, row in top.iterrows():
    fig.add_trace(go.Scatter(x=list(evo.columns), y=row.values, mode='lines+markers', name=team))
fig.update_layout(title='Probabilidad de campeón con los resultados reales fijados (%)',
                  yaxis_title='%', height=500)
fig.show()

## 4. Ranking después del Mundial

In [ ]:
teams = [t for g in ctx.fixtures for t in g[1:3]]
post.ratings_table(sorted(set(teams))).head(20).round(1)

## 5. Simular desde cualquier punto del torneo
Cambia `hasta` por 'GROUP', 'R32', 'R16', 'QF' o 'SF'.

In [ ]:
from modelo.data import known_results_from
hasta = ['GROUP', 'R32']
known = known_results_from(ctx.wc[ctx.wc['stage'].isin(hasta)])
TournamentSimulator(pre, group_fixtures=ctx.fixtures, known=known, seed=1).run(5000)\
    .head(10)[['team', 'Cuartos (QF) %', 'Semifinal (SF) %', 'Final (F) %', 'Campeón %']].round(1)